In [135]:
import pickle as pkl
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import brown
from nltk import download
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import random
from tqdm import tqdm

torch.manual_seed(42)

In [136]:
import importlib
import enc_dec_lstm
importlib.reload(enc_dec_lstm)
from enc_dec_lstm import Encoder, Decoder, Encoder_Decoder_Model

In [137]:
device = "xpu" if torch.xpu.is_available() else "cpu"
device

'xpu'

In [138]:
with open('../data/train_data.pkl', 'rb') as f:
    train_data = pkl.load(f)

with open('../data/val_data.pkl', 'rb') as f:
    val_data = pkl.load(f)

In [139]:
word_counts = Counter(w for sent in train_data for w, _ in sent)
tag_counts = Counter(t for sent in train_data for _, t in sent)

word2idx = {w: i+2 for i, (w, _) in enumerate(word_counts.items()) if word_counts[w] > 1}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1
idx2word = {i: w for w, i in word2idx.items()}

tag2idx = {t: i+2 for i, (t, _) in enumerate(tag_counts.items())}
tag2idx["<PAD>"] = 0
tag2idx["<SOS>"] = 1
idx2tag = {i: t for t, i in tag2idx.items()}

In [140]:
len(word2idx)

24756

In [142]:
vocab_size = len(word2idx)
tag_size = len(tag2idx)

In [143]:
MAX_LEN = 50

In [144]:
class POSTagDataset(Dataset):
    def __init__(self, sentences):
        self.data = []
        for sent in sentences:
            words, tags = zip(*sent)
            # if len(words) > MAX_LEN:
            #     words = words[:MAX_LEN]
            #     tags = tags[:MAX_LEN]
            # else:
            #     words = list(words) + ["<PAD>"] * (MAX_LEN - len(words))
            #     tags = list(tags) + ["<PAD>"] * (MAX_LEN - len(tags))
            
            word_ids = [word2idx.get(w, 1) for w in words]
            tag_ids = [tag2idx.get(t, 0) if t in tag2idx else 0 for t in tags]

            length = len(word_ids)

            self.data.append((torch.tensor(word_ids).to(device), torch.tensor(tag_ids).to(device), length))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [145]:
def data_collate_fn(batch):
    words, tags, lens = zip(*batch)
    words_batch = nn.utils.rnn.pad_sequence(words, batch_first=True, padding_value=0).to(device)
    tags_batch = nn.utils.rnn.pad_sequence(tags, batch_first=True, padding_value=0).to(device)
    return words_batch, tags_batch, lens

In [146]:
train_dataset = POSTagDataset(train_data)
val_dataset = POSTagDataset(val_data)

In [147]:
batch_size=128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collate_fn)

In [148]:
epochs = 25
embed_dim = 64
hidden_dim = 128
model=Encoder_Decoder_Model(vocab_size, embed_dim, hidden_dim, tag_size, tag2idx["<SOS>"]).to(device)
lr=0.001
optimizer = optim.Adam(model.parameters(), lr=lr)
# criterion = nn.CrossEntropyLoss()
criterion = nn.CrossEntropyLoss(ignore_index=0)

for e in range(epochs):
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {e+1}/{epochs}", total=len(train_loader)):
        words_batch, tags_batch, length_batch = batch
        input_seq = words_batch
        # input_tags = torch.cat([torch.tensor([[tag2idx["<SOS>"]]]*len(input_seq), device=device), tags_batch[:, :-1]], dim=1)
        output_tags = tags_batch
        # outputs = model(input_seq, input_tags)
        outputs = model(input_seq, output_tags, length_batch)
        pred_logits = outputs[0]
        loss = criterion(pred_logits.view(-1, tag_size), output_tags.view(-1))
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Training Loss: {train_loss / len(train_loader)}")

    with torch.no_grad():
        val_loss = 0.0
        for batch in tqdm(val_loader, desc=f"Validation {e+1}/{epochs}", total=len(val_loader)):
            words_batch, tags_batch, length_batch = batch
            input_seq = words_batch
            # input_tags = torch.cat([torch.tensor([[tag2idx["<SOS>"]]]*len(input_seq), device=device), tags_batch[:, :-1]], dim=1)
            output_tags = tags_batch
            # outputs = model(input_seq, input_tags)
            outputs = model(input_seq, output_tags, length_batch)
            pred_logits = outputs[0]
            loss = criterion(pred_logits.view(-1, tag_size), output_tags.view(-1))
            val_loss += loss.item()
        print(f"Validation Loss: {val_loss / len(val_loader)}")

    model_path = f'models/encoder_decoder_model_{lr}lr_{batch_size}bs_{e+1}epochs.pth'
    torch.save(model.state_dict(), model_path)

Epoch 1/25:   0%|          | 0/359 [00:00<?, ?it/s]

Epoch 1/25: 100%|██████████| 359/359 [00:41<00:00,  8.63it/s]


Training Loss: 1.6497418870500868


Validation 1/25: 100%|██████████| 45/45 [00:01<00:00, 36.21it/s]


Validation Loss: 1.4600637197494506


...

Validation Loss: 0.628666529390547


Epoch 25/25: 100%|██████████| 359/359 [00:42<00:00,  8.36it/s]


Training Loss: 0.5184512024636388


Validation 25/25: 100%|██████████| 45/45 [00:01<00:00, 37.59it/s]


Validation Loss: 0.6190310266282824
